This is the notebook to visualize and check the dataset.

In [1]:
import torch

# Check if GPU is available
gpu_available = torch.cuda.is_available()
print(f"Is GPU available? {gpu_available}")

ModuleNotFoundError: No module named 'torch'

In [5]:
import os
from pathlib import Path
import pandas as pd
import nibabel as nib
import numpy as np

def explore_adni_directory(base_path):
    """
    Explore and analyze the ADNI directory structure and files
    
    Args:
        base_path (str): Path to the ADNI directory
    """
    # Convert to Path object for easier handling
    base_dir = Path(base_path)
    
    # Initialize data collection
    file_info = []
    
    # Walk through directory
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith(('.nii', '.nii.gz')):
                full_path = Path(root) / file
                relative_path = full_path.relative_to(base_dir)
                
                try:
                    # Load NIfTI file
                    img = nib.load(str(full_path))
                    data = img.get_fdata()
                    
                    file_info.append({
                        'filename': file,
                        'relative_path': str(relative_path),
                        'size_mb': full_path.stat().st_size / (1024 * 1024),
                        'shape': data.shape,
                        'data_type': str(data.dtype),
                        'min_value': np.min(data),
                        'max_value': np.max(data),
                        'mean_value': np.mean(data)
                    })
                except Exception as e:
                    print(f"Error processing {file}: {str(e)}")
    
    # Convert to DataFrame for easier analysis
    df = pd.DataFrame(file_info)
    
    # Print summary
    print("\n=== ADNI Directory Analysis ===")
    print(f"\nBase Path: {base_path}")
    print(f"Total number of NIfTI files: {len(df)}")
    print("\nDirectory Structure:")
    print_directory_tree(base_dir)
    
    print("\nFile Statistics:")
    print(f"Average file size: {df['size_mb'].mean():.2f} MB")
    print(f"Total size: {df['size_mb'].sum():.2f} MB")
    
    if not df.empty:
        print("\nImage Dimensions:")
        print(df['shape'].value_counts().to_string())
        
        print("\nData Types:")
        print(df['data_type'].value_counts().to_string())
    
    return df

def print_directory_tree(path, level=0, max_level=3):
    """
    Print directory tree structure
    """
    if level > max_level:
        return
        
    indent = "  " * level
    path_obj = Path(path)
    
    try:
        # Print current directory
        print(f"{indent}└── {path_obj.name}/")
        
        # Recursively print subdirectories
        for item in path_obj.iterdir():
            if item.is_dir():
                print_directory_tree(item, level + 1, max_level)
            elif level == max_level and item.suffix in ['.nii', '.nii.gz']:
                print(f"{indent}    └── {item.name}")
    except PermissionError:
        print(f"{indent}    ⚠️  Permission denied")

def analyze_sample_image(file_path):
    """
    Analyze a sample NIfTI image in detail
    """
    img = nib.load(file_path)
    data = img.get_fdata()
    
    print(f"\n=== Sample Image Analysis: {Path(file_path).name} ===")
    print(f"Dimensions: {data.shape}")
    print(f"Data type: {data.dtype}")
    print(f"Value range: [{np.min(data):.2f}, {np.max(data):.2f}]")
    print(f"Mean value: {np.mean(data):.2f}")
    print(f"Standard deviation: {np.std(data):.2f}")
    print("\nHeader information:")
    print(img.header)
    
    return img, data

# Main execution
if __name__ == "__main__":
    adni_path = r"E:/KHU Gangdong Hospital Data/PET1_FDG/ADNI1"
    df = explore_adni_directory(adni_path)


=== ADNI Directory Analysis ===

Base Path: E:/KHU Gangdong Hospital Data/PET1_FDG/ADNI1
Total number of NIfTI files: 5777

Directory Structure:
└── ADNI1/

File Statistics:
Average file size: 0.90 MB
Total size: 5208.41 MB

Image Dimensions:
shape
(79, 95, 79)    5777

Data Types:
data_type
float64    5777


In [7]:
# Lets first analyze the dataset.

import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def analyze_dataset_structure(adni_path_path):
    """
    Analyze the structure of the ADNI dataset directories
    """
    dataset_info = {
        'total_files': 0,
        'file_types': set(),
        'subfolder_count': 0,
        'subject_count': set()
    }
    
    for root, dirs, files in os.walk(adni_path):
        dataset_info['subfolder_count'] += len(dirs)
        for file in files:
            if file.endswith('.nii') or file.endswith('.nii.gz'):
                dataset_info['total_files'] += 1
                dataset_info['file_types'].add(file.split('.')[-1])
                # Assuming subject IDs are part of the filename
                subject_id = extract_subject_id(file)  # You'll need to implement this based on your naming convention
                if subject_id:
                    dataset_info['subject_count'].add(subject_id)
    
    return dataset_info

def load_and_inspect_nifti(file_path):
    """
    Load and return basic information about a NIfTI file
    """
    img = nib.load(file_path)
    data = img.get_fdata()
    
    info = {
        'shape': data.shape,
        'data_type': data.dtype,
        'affine': img.affine,
        'header': img.header,
        'value_range': (np.min(data), np.max(data)),
        'mean': np.mean(data),
        'std': np.std(data)
    }
    
    return info, data

def visualize_slice(data, slice_num=None, axis=2):
    """
    Visualize a single slice from the 3D volume
    """
    if slice_num is None:
        slice_num = data.shape[axis] // 2
        
    if axis == 0:
        slice_data = data[slice_num, :, :]
    elif axis == 1:
        slice_data = data[:, slice_num, :]
    else:
        slice_data = data[:, :, slice_num]
    
    plt.figure(figsize=(10, 10))
    plt.imshow(slice_data, cmap='gray')
    plt.colorbar()
    plt.title(f'Slice {slice_num} along axis {axis}')
    plt.show()

def batch_process_folder(folder_path):
    """
    Process all NIfTI files in a folder and return summary statistics
    """
    summary_stats = []
    for file in os.listdir(folder_path):
        if file.endswith(('.nii', '.nii.gz')):
            file_path = os.path.join(folder_path, file)
            info, _ = load_and_inspect_nifti(file_path)
            summary_stats.append({
                'file': file,
                'shape': info['shape'],
                'value_range': info['value_range'],
                'mean': info['mean'],
                'std': info['std']
            })
    
    return summary_stats

# Example usage function
def analyze_adni_folder(adni_path):
    """
    Analyze an ADNI folder and print summary information
    """
    print(f"Analyzing ADNI folder: {adni_path}")
    
    # Get dataset structure information
    structure_info = analyze_dataset_structure(adni_path)
    print("\nDataset Structure:")
    print(f"Total NIfTI files: {structure_info['total_files']}")
    print(f"File types present: {structure_info['file_types']}")
    print(f"Number of subfolders: {structure_info['subfolder_count']}")
    print(f"Number of unique subjects: {len(structure_info['subject_count'])}")
    
    # Process the first few files as an example
    print("\nProcessing sample files:")
    sample_stats = batch_process_folder(adni_path)
    if sample_stats:
        print("\nSample file statistics:")
        for stat in sample_stats[:5]:  # Show first 5 files
            print(f"\nFile: {stat['file']}")
            print(f"Shape: {stat['shape']}")
            print(f"Value range: {stat['value_range']}")
            print(f"Mean: {stat['mean']:.2f}")
            print(f"Std: {stat['std']:.2f}")